# OMNet: Breast Cancer Histopathology Classification

**OMNet** is a research-oriented deep learning framework for binary classification of breast cancer histopathology images (benign vs. malignant) using a custom CNN architecture trained **entirely from scratch** on the **BreakHis 400X** dataset.

---

| Property | Value |
|---|---|
| **Architecture** | OMNet-V1 (Custom 4-block CNN) |
| **Dataset** | BreakHis 400X — 1,693 images at 400× magnification |
| **Task** | Binary classification (Benign vs. Malignant) |
| **Framework** | PyTorch |
| **Hardware** | Google Colab T4/L4 GPU |
| **Pretrained Weights** | None — trained from scratch |

### Research Objectives
1. Train a custom CNN from scratch — no transfer learning
2. Achieve robust binary classification on histopathology images
3. Produce comprehensive evaluation metrics and visualizations
4. Build modular interfaces for future integration of Vision Transformers, Deep Metric Learning, and Proxy Anchor Loss

### Notebook Structure
This notebook contains **23 sequential sections**, each with Purpose, Theory, Implementation, and Expected Output. Execute all cells **top-to-bottom** without modification.

> **Smoke Test Mode**: By default, `CONFIG['smoke_test'] = True` runs only 2 epochs for pipeline validation. Set to `False` for full training.

# Section 1: Environment Setup

**Purpose:** Install dependencies and import all libraries required for the pipeline.

**Theory:** We use PyTorch as the deep learning framework, scikit-learn for evaluation metrics, matplotlib/seaborn for visualization, and kagglehub for dataset download. `pytorch-grad-cam` provides Gradient-weighted Class Activation Mapping for model interpretability.

**Expected Output:** Version numbers for PyTorch and Python, confirmation of successful imports.

In [ ]:
# ============================================================
# Section 1: Environment Setup
# ============================================================
# Install packages not pre-installed on Colab
!pip install -q kagglehub grad-cam

# --- Standard Library ---
import os
import sys
import json
import time
import random
import warnings
import gc
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter, OrderedDict

# --- Scientific Computing ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# --- Image Processing ---
from PIL import Image

# --- Progress Bars ---
from tqdm.auto import tqdm

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# --- Metrics ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.manifold import TSNE

# --- Suppress Warnings ---
warnings.filterwarnings('ignore')

# --- Plotting Style ---
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})
sns.set_theme(style='whitegrid', palette='husl')

# --- Reproducibility ---
def set_seed(seed=42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

print("=" * 60)
print("  OMNet Environment Setup Complete")
print("=" * 60)
print(f"  PyTorch   : {torch.__version__}")
print(f"  Python    : {sys.version.split()[0]}")
print(f"  NumPy     : {np.__version__}")
print(f"  Pandas    : {pd.__version__}")
print(f"  CUDA Avail: {torch.cuda.is_available()}")
print("=" * 60)

# Section 2: GPU Detection

**Purpose:** Detect available GPU hardware and configure the compute device.

**Theory:** Deep learning training benefits enormously from GPU acceleration. Google Colab provides NVIDIA T4 (16 GB) or L4 GPUs. We detect the hardware and set a global `DEVICE` variable used throughout the notebook.

**Expected Output:** GPU name, memory, and compute capability — or a CPU fallback warning.

In [ ]:
# ============================================================
# Section 2: GPU Detection
# ============================================================
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    gpu_props = torch.cuda.get_device_properties(0)
    print("=" * 60)
    print("  GPU Detected!")
    print("=" * 60)
    print(f"  Device    : {gpu_props.name}")
    print(f"  Memory    : {gpu_props.total_mem / 1e9:.1f} GB")
    print(f"  Compute   : {gpu_props.major}.{gpu_props.minor}")
    print(f"  CUDA Ver  : {torch.version.cuda}")
    print(f"  cuDNN Ver : {torch.backends.cudnn.version()}")
    print("=" * 60)
else:
    DEVICE = torch.device('cpu')
    print("\u26a0\ufe0f  No GPU detected. Using CPU.")
    print("   Training will be significantly slower.")

print(f"\nActive device: {DEVICE}")

# Section 3: Mount Google Drive

**Purpose:** Mount Google Drive for persistent storage of models, logs, and outputs. If running locally (not on Colab), outputs are saved to the current directory.

**Theory:** Colab VMs are ephemeral — all local files are lost when the runtime disconnects. Google Drive provides persistent storage that survives disconnections.

**Expected Output:** Confirmation of mount, output directory paths created.

In [ ]:
# ============================================================
# Section 3: Mount Google Drive
# ============================================================
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/OMNet')
else:
    PROJECT_ROOT = Path('.')

# --- Output directories ---
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
GRADCAM_DIR = OUTPUT_DIR / 'gradcam'
MISCLASSIFIED_DIR = OUTPUT_DIR / 'misclassified'
CORRECT_PRED_DIR = OUTPUT_DIR / 'correct_predictions'

for d in [OUTPUT_DIR, GRADCAM_DIR, MISCLASSIFIED_DIR, CORRECT_PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("  Storage Configuration")
print("=" * 60)
print(f"  Running on : {'Google Colab' if IN_COLAB else 'Local Machine'}")
print(f"  Project Root : {PROJECT_ROOT}")
print(f"  Output Dir   : {OUTPUT_DIR}")
print("=" * 60)

# --- Utility function for saving figures ---
def save_figure(fig, filename, dpi=150):
    """Save a matplotlib figure to the output directory and display it."""
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"  \u2713 Saved: {path}")

# Hyperparameter Configuration

> **IMPORTANT:** All configurable hyperparameters are centralized here. Modify **only this cell** to change the experiment configuration, ensuring reproducibility and consistency across runs.

Change `smoke_test` to `False` and `epochs` to your desired count for full training.

In [ ]:
# ============================================================
# IMPORTANT: All configurable hyperparameters are centralized
# here. Modify ONLY this cell to change the experiment
# configuration, ensuring reproducibility and consistency.
# ============================================================

CONFIG = {
    # --- Data ---
    "data": {
        "dataset_id": "pankaj4321/breakhis-400x",
        "input_size": 224,
        "batch_size": 32,
        "num_workers": 2,
        "pin_memory": True,
    },
    # --- Model ---
    "model": {
        "name": "omnet_v1",
        "num_classes": 2,
        "embedding_dim": 128,
        "dropout": 0.5,
        "block_dropout": 0.25,
    },
    # --- Training ---
    "training": {
        "epochs": 50,
        "optimizer": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "scheduler": "cosine",       # "cosine" | "plateau"
        "early_stopping_patience": 10,
        "mixed_precision": True,
    },
    # --- Loss ---
    "loss": {
        "name": "cross_entropy",     # "cross_entropy" | "focal"
        "focal_gamma": 2.0,
        "focal_alpha": None,         # None = auto-computed from class weights
        "label_smoothing": 0.05,
    },
    # --- Normalization ---
    "normalization": "dataset",      # "dataset" | "imagenet"
    # --- Reproducibility ---
    "seed": 42,
    # --- Smoke Test ---
    "smoke_test": True,              # True = 2 epochs for pipeline validation
}

# Apply seed
set_seed(CONFIG['seed'])

# Print configuration
print("=" * 60)
print("  Experiment Configuration")
print("=" * 60)
for section, params in CONFIG.items():
    if isinstance(params, dict):
        print(f"\n  [{section}]")
        for k, v in params.items():
            print(f"    {k:30s}: {v}")
    else:
        print(f"  {section:32s}: {params}")
print("\n" + "=" * 60)
if CONFIG['smoke_test']:
    print("  \u26a0\ufe0f  SMOKE TEST MODE: Training limited to 2 epochs")
    print("     Set CONFIG['smoke_test'] = False for full training")
    print("=" * 60)

#  Section 4: Dataset Loading

**Purpose:** Download the BreakHis 400X dataset from Kaggle using `kagglehub` and verify its directory structure.

**Theory:** The BreakHis (Breast Cancer Histopathological Image) dataset contains microscopy images of breast tumor tissue at 400× magnification. The Kaggle version (`pankaj4321/breakhis-400x`) is pre-split into `train/`, `validation/`, and `test/` directories, each containing `benign/` and `malignant/` subdirectories. We use the existing splits as-is — no re-splitting.

**Expected Output:** Dataset path, confirmation of directory structure, file counts per split.

In [ ]:
# ============================================================
# Section 4: Dataset Loading
# ============================================================
import kagglehub

# Set Kaggle API key
# For security in production, use Colab Secrets: userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_KEY'] = 'KGAT_f7a272362f7109cb6b9ed2652882282f'

# Download dataset
print("Downloading BreakHis 400X dataset...")
data_path = Path(kagglehub.dataset_download(CONFIG['data']['dataset_id']))
print(f"Downloaded to: {data_path}")

# Find the actual data root (handles nested directory structures)
def find_data_root(path):
    """Locate the directory containing train/validation/test splits."""
    if (path / 'train').exists():
        return path
    for item in sorted(path.rglob('train')):
        if item.is_dir() and (item.parent / 'test').exists():
            return item.parent
    raise FileNotFoundError(f"Cannot find 'train' directory in {path}")

DATA_ROOT = find_data_root(data_path)

# Detect validation directory name (some datasets use 'val' vs 'validation')
VAL_SPLIT = 'validation' if (DATA_ROOT / 'validation').exists() else 'val'
SPLITS = ['train', VAL_SPLIT, 'test']
CLASS_NAMES = ['benign', 'malignant']

# Verify directory structure
print("\n" + "=" * 60)
print("  Dataset Structure Verification")
print("=" * 60)
print(f"  Data root: {DATA_ROOT}")

all_ok = True
for split in SPLITS:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        print(f"  \u274c  Missing: {split}/")
        all_ok = False
        continue
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if not cls_dir.exists():
            print(f"  \u274c  Missing: {split}/{cls}/")
            all_ok = False
        else:
            n_files = len(list(cls_dir.glob('*')))
            print(f"  \u2713  {split}/{cls}/ \u2014 {n_files} images")

if all_ok:
    print("\n  \u2705 All directories verified successfully!")
else:
    print("\n  \u274c Some directories are missing. Check the dataset.")
print("=" * 60)

# Section 5: Dataset Analysis

**Purpose:** Compute detailed statistics about the dataset: image counts, class ratios, and image resolutions.

**Theory:** Understanding the data distribution is critical before training. Class imbalance (BreakHis has ~67.7% malignant, ~32.3% benign) directly impacts model performance and requires mitigation strategies like weighted loss functions.

**Expected Output:** Summary table with counts and ratios per split, resolution verification.

In [ ]:
# ============================================================
# Section 5: Dataset Analysis
# ============================================================
print("Analyzing dataset...")

# Collect statistics
analysis_data = []
resolution_set = set()
corrupt_files = []

for split in SPLITS:
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = DATA_ROOT / split / cls_name
        image_files = sorted(list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpg')) +
                             list(cls_dir.glob('*.jpeg')) + list(cls_dir.glob('*.bmp')))
        count = 0
        for img_path in image_files:
            try:
                with Image.open(img_path) as img:
                    resolution_set.add(img.size)
                    count += 1
            except Exception as e:
                corrupt_files.append((str(img_path), str(e)))
        analysis_data.append({
            'Split': split,
            'Class': cls_name,
            'Count': count,
        })

# Build summary DataFrame
df_analysis = pd.DataFrame(analysis_data)
df_pivot = df_analysis.pivot(index='Split', columns='Class', values='Count').fillna(0).astype(int)
df_pivot['Total'] = df_pivot.sum(axis=1)
df_pivot['Benign %'] = (df_pivot['benign'] / df_pivot['Total'] * 100).round(1)
df_pivot['Malignant %'] = (df_pivot['malignant'] / df_pivot['Total'] * 100).round(1)
df_pivot['Ratio (M:B)'] = (df_pivot['malignant'] / df_pivot['benign']).round(2)

print("\n" + "=" * 60)
print("  Dataset Summary")
print("=" * 60)
print(df_pivot.to_string())
print(f"\n  Total images across all splits: {df_pivot['Total'].sum()}")
print(f"  Unique resolutions found: {resolution_set}")
if corrupt_files:
    print(f"  \u26a0\ufe0f  Corrupt files found: {len(corrupt_files)}")
    for path, err in corrupt_files[:5]:
        print(f"      {path}: {err}")
else:
    print(f"  \u2713 No corrupt files detected")
print("=" * 60)

# Section 6: Dataset Visualization

**Purpose:** Visualize sample images from each class and the class distribution across splits.

**Theory:** Visual inspection of histopathology images reveals key morphological differences between benign and malignant tissue. Benign images typically show well-organized cellular structures, while malignant images exhibit irregular cell shapes, increased nuclear-to-cytoplasmic ratio, and disordered tissue architecture.

**Expected Output:** Sample image grid (benign vs. malignant) and class distribution bar chart.

In [ ]:
# ============================================================
# Section 6: Dataset Visualization
# ============================================================

# --- 6a: Sample Images Grid ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('BreakHis 400X Sample Images', fontsize=16, fontweight='bold', y=1.02)

for row_idx, cls_name in enumerate(CLASS_NAMES):
    cls_dir = DATA_ROOT / 'train' / cls_name
    sample_files = sorted(list(cls_dir.glob('*.png')))[:4]
    for col_idx, img_path in enumerate(sample_files):
        img = Image.open(img_path).convert('RGB')
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(
            f'{cls_name.capitalize()}', fontsize=11,
            color='green' if cls_name == 'benign' else 'red'
        )
        axes[row_idx, col_idx].axis('off')

plt.tight_layout()
save_figure(fig, 'sample_images.png')
plt.show()

# --- 6b: Class Distribution Bar Chart ---
fig, ax = plt.subplots(figsize=(10, 6))

split_labels = SPLITS
benign_counts = [df_pivot.loc[s, 'benign'] if s in df_pivot.index else 0 for s in split_labels]
malignant_counts = [df_pivot.loc[s, 'malignant'] if s in df_pivot.index else 0 for s in split_labels]

x = np.arange(len(split_labels))
width = 0.35

bars1 = ax.bar(x - width/2, benign_counts, width, label='Benign', color='#2ecc71', edgecolor='white')
bars2 = ax.bar(x + width/2, malignant_counts, width, label='Malignant', color='#e74c3c', edgecolor='white')

ax.set_xlabel('Dataset Split')
ax.set_ylabel('Number of Images')
ax.set_title('Class Distribution Across Splits', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in split_labels])
ax.legend()

# Add count labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
save_figure(fig, 'class_distribution.png')
plt.show()

# Section 7: Dataset Statistics

**Purpose:** Compute the dataset-specific RGB mean and standard deviation using all training images.

> **IMPORTANT:** We compute dataset-specific normalization statistics instead of using ImageNet defaults. This better matches the BreakHis data distribution (histopathology staining patterns differ significantly from natural images) and improves experiment reproducibility.

**Theory:** Normalization transforms pixel values to have zero mean and unit variance per channel. Using dataset-specific statistics ensures the model receives inputs centered on the actual data distribution, leading to faster convergence and potentially better performance.

**Expected Output:** Per-channel (R, G, B) mean and standard deviation values.

In [ ]:
# ============================================================
# Section 7: Dataset Statistics
# ============================================================
# IMPORTANT: Compute the dataset-specific RGB mean and
# standard deviation using all training images. Use these
# values for normalization instead of the default ImageNet
# statistics to better match the BreakHis data distribution
# and improve experiment reproducibility.
# ============================================================

print("Computing dataset-specific normalization statistics...")
print("(Iterating over all training images)\n")

# Collect all training image paths
train_dir = DATA_ROOT / 'train'
train_image_paths = sorted(
    list(train_dir.glob('*/*.png')) + list(train_dir.glob('*/*.jpg')) +
    list(train_dir.glob('*/*.jpeg'))
)

# Compute channel-wise statistics using Welford's online algorithm
pixel_sum = torch.zeros(3, dtype=torch.float64)
pixel_sq_sum = torch.zeros(3, dtype=torch.float64)
total_pixels = 0

to_tensor = transforms.ToTensor()  # Converts [0,255] uint8 -> [0,1] float

for img_path in tqdm(train_image_paths, desc='Computing stats'):
    img = Image.open(img_path).convert('RGB')
    tensor = to_tensor(img)  # (3, H, W) in [0, 1]
    pixel_sum += tensor.sum(dim=[1, 2]).double()
    pixel_sq_sum += (tensor ** 2).sum(dim=[1, 2]).double()
    total_pixels += tensor.shape[1] * tensor.shape[2]

# Calculate mean and std
DATASET_MEAN = (pixel_sum / total_pixels).float().tolist()
DATASET_STD = torch.sqrt(pixel_sq_sum / total_pixels - (pixel_sum / total_pixels) ** 2).float().tolist()

# ImageNet statistics (for optional comparison experiments)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Select normalization based on CONFIG
if CONFIG['normalization'] == 'dataset':
    NORM_MEAN = DATASET_MEAN
    NORM_STD = DATASET_STD
    norm_source = 'Dataset-specific'
else:
    NORM_MEAN = IMAGENET_MEAN
    NORM_STD = IMAGENET_STD
    norm_source = 'ImageNet'

print("\n" + "=" * 60)
print("  Normalization Statistics")
print("=" * 60)
print(f"  Dataset Mean  : R={DATASET_MEAN[0]:.4f}  G={DATASET_MEAN[1]:.4f}  B={DATASET_MEAN[2]:.4f}")
print(f"  Dataset Std   : R={DATASET_STD[0]:.4f}  G={DATASET_STD[1]:.4f}  B={DATASET_STD[2]:.4f}")
print(f"  ImageNet Mean : R={IMAGENET_MEAN[0]:.4f}  G={IMAGENET_MEAN[1]:.4f}  B={IMAGENET_MEAN[2]:.4f}")
print(f"  ImageNet Std  : R={IMAGENET_STD[0]:.4f}  G={IMAGENET_STD[1]:.4f}  B={IMAGENET_STD[2]:.4f}")
print(f"\n  \u27a1 Using: {norm_source} normalization")
print(f"  Active Mean   : {[round(v, 4) for v in NORM_MEAN]}")
print(f"  Active Std    : {[round(v, 4) for v in NORM_STD]}")
print("=" * 60)

# --- Visualize channel distributions ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['#e74c3c', '#2ecc71', '#3498db']

# Sample 50 images for the histogram
sample_paths = random.sample(train_image_paths, min(50, len(train_image_paths)))
for ch_idx in range(3):
    all_values = []
    for p in sample_paths:
        img_t = to_tensor(Image.open(p).convert('RGB'))
        all_values.extend(img_t[ch_idx].flatten().numpy())
    axes[ch_idx].hist(all_values, bins=50, color=channel_colors[ch_idx], alpha=0.7, edgecolor='white')
    axes[ch_idx].axvline(DATASET_MEAN[ch_idx], color='black', linestyle='--', linewidth=2, label=f'Mean={DATASET_MEAN[ch_idx]:.3f}')
    axes[ch_idx].set_title(f'{channel_names[ch_idx]} Channel', fontweight='bold')
    axes[ch_idx].set_xlabel('Pixel Value')
    axes[ch_idx].set_ylabel('Frequency')
    axes[ch_idx].legend()

fig.suptitle('RGB Channel Distributions (Training Set)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
save_figure(fig, 'channel_distributions.png')
plt.show()

# Section 8: DataLoader

**Purpose:** Define the `BreakHisDataset` class, build transform pipelines, create DataLoaders, and compute class weights for balanced training.

**Theory:** PyTorch's `Dataset` abstraction provides a standardized interface for loading data. We define transforms for training (with augmentation) and evaluation (without). Class weights are computed inversely proportional to class frequency to mitigate the 2:1 malignant-to-benign imbalance.

**Expected Output:** DataLoader creation confirmation, batch shape verification, class weights.

In [ ]:
# ============================================================
# Section 8: DataLoader
# ============================================================
IMG_SIZE = CONFIG['data']['input_size']

# --- Transform Pipelines ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),      # Resize to 256
    transforms.CenterCrop(IMG_SIZE),       # Center crop to 224
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


class BreakHisDataset(Dataset):
    """
    BreakHis 400X Dataset for binary classification.

    Loads images from benign/ and malignant/ subdirectories.
    Labels: 0 = benign, 1 = malignant

    This class provides the interface needed for both standard
    classification and future metric learning experiments.
    """
    CLASSES = ['benign', 'malignant']

    def __init__(self, root_dir, split, transform=None):
        self.root = Path(root_dir) / split
        self.transform = transform
        self.samples = []   # List of Path objects
        self.labels = []    # List of int labels

        for label_idx, class_name in enumerate(self.CLASSES):
            class_dir = self.root / class_name
            if not class_dir.exists():
                continue
            image_files = sorted(
                list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpg')) +
                list(class_dir.glob('*.jpeg'))
            )
            for img_path in image_files:
                self.samples.append(img_path)
                self.labels.append(label_idx)

        self.labels = np.array(self.labels)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_class_weights(self):
        """Compute inverse-frequency class weights for balanced training."""
        counts = Counter(self.labels.tolist())
        total = len(self.labels)
        weights = torch.tensor([total / (len(counts) * counts[i]) for i in range(len(counts))],
                               dtype=torch.float32)
        return weights


# --- Create Datasets ---
train_dataset = BreakHisDataset(DATA_ROOT, 'train', transform=train_transform)
val_dataset = BreakHisDataset(DATA_ROOT, VAL_SPLIT, transform=eval_transform)
test_dataset = BreakHisDataset(DATA_ROOT, 'test', transform=eval_transform)

# --- Compute Class Weights ---
class_weights = train_dataset.get_class_weights().to(DEVICE)

# --- Create DataLoaders ---
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['data']['batch_size'],
    shuffle=True,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=CONFIG['data']['pin_memory'],
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['data']['batch_size'],
    shuffle=False,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=CONFIG['data']['pin_memory'],
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['data']['batch_size'],
    shuffle=False,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=CONFIG['data']['pin_memory'],
)

# --- Verify ---
print("=" * 60)
print("  DataLoader Summary")
print("=" * 60)
print(f"  Train : {len(train_dataset):5d} images | {len(train_loader):3d} batches")
print(f"  Val   : {len(val_dataset):5d} images | {len(val_loader):3d} batches")
print(f"  Test  : {len(test_dataset):5d} images | {len(test_loader):3d} batches")
print(f"  Batch : {CONFIG['data']['batch_size']}")
print(f"  Class Weights : {class_weights.cpu().numpy().round(4)}")

# Verify batch shape
sample_images, sample_labels = next(iter(train_loader))
print(f"\n  Batch shape  : {sample_images.shape}")
print(f"  Label shape  : {sample_labels.shape}")
print(f"  Pixel range  : [{sample_images.min():.3f}, {sample_images.max():.3f}]")
print("=" * 60)

# Section 9: Data Augmentation

**Purpose:** Visualize the effect of training augmentations on sample images.

**Theory:** Data augmentation artificially increases training set diversity, reducing overfitting. For histopathology, geometric transforms (flips, rotations, crops) are particularly effective because tissue orientation is arbitrary under the microscope. Color jitter simulates variations in staining protocols across different labs.

**Expected Output:** Grid showing one original image alongside 8 augmented variants.

In [ ]:
# ============================================================
# Section 9: Data Augmentation Visualization
# ============================================================

# Select a sample image for augmentation demonstration
sample_path = train_dataset.samples[0]
original_img = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
fig.suptitle('Data Augmentation Examples', fontsize=16, fontweight='bold', y=1.02)

# Original image (top-left)
axes[0, 0].imshow(original_img)
axes[0, 0].set_title('Original', fontsize=12, fontweight='bold', color='blue')
axes[0, 0].axis('off')

# 8 augmented versions
for i in range(8):
    row, col = divmod(i + 1, 3)
    # Apply training transform
    augmented = train_transform(original_img)
    # Denormalize for display
    denorm = augmented.clone()
    for c in range(3):
        denorm[c] = denorm[c] * NORM_STD[c] + NORM_MEAN[c]
    denorm = denorm.clamp(0, 1)
    axes[row, col].imshow(denorm.permute(1, 2, 0).numpy())
    axes[row, col].set_title(f'Augmented #{i+1}', fontsize=11)
    axes[row, col].axis('off')

plt.tight_layout()
save_figure(fig, 'augmentation_examples.png')
plt.show()

# Print augmentation pipeline
print("\nTraining Augmentation Pipeline:")
for i, t in enumerate(train_transform.transforms):
    print(f"  {i+1}. {t}")

# Section 10: CNN Architecture (OMNet-V1)

**Purpose:** Define the OMNet-V1 custom CNN architecture for binary classification.

**Theory:** OMNet-V1 is a 4-block convolutional neural network designed specifically for histopathology image classification. Each convolutional block uses two 3×3 convolutions with batch normalization and ReLU activation, followed by max pooling and dropout for regularization. The architecture progressively increases channel depth (32 → 64 → 128 → 256) to capture increasingly abstract features.

**Architecture:**
```
Block 1: Conv(3→32) → BN → ReLU → Conv(32→32) → BN → ReLU → MaxPool → Drop(0.25)
Block 2: Conv(32→64) → BN → ReLU → Conv(64→64) → BN → ReLU → MaxPool → Drop(0.25)
Block 3: Conv(64→128) → BN → ReLU → Conv(128→128) → BN → ReLU → MaxPool → Drop(0.25)
Block 4: Conv(128→256) → BN → ReLU → Conv(256→256) → BN → ReLU → GAP
Embedding: Linear(256→128) → ReLU → Dropout(0.5)
Classifier: Linear(128→2)
```

**Future Compatibility:** The model exposes `extract_features()` and `get_embedding_dim()` methods, enabling drop-in replacement with Vision Transformers or integration with Deep Metric Learning / Proxy Anchor Loss without modifying the training pipeline.

**Expected Output:** Model class definition (no output until Section 11).

In [ ]:
# ============================================================
# Section 10: CNN Architecture - OMNet-V1
# ============================================================

class ConvBlock(nn.Module):
    """
    A convolutional block consisting of two 3x3 convolutions,
    each followed by batch normalization and ReLU activation,
    then max pooling and spatial dropout.
    """
    def __init__(self, in_channels, out_channels, dropout_rate=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout2d(p=dropout_rate)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = F.relu(self.bn2(self.conv2(x)), inplace=True)
        x = self.pool(x)
        x = self.dropout(x)
        return x


class OMNetV1(nn.Module):
    """
    OMNet-V1: Custom 4-block CNN for histopathology classification.

    Trained entirely from scratch. Provides a modular interface for:
      - forward(x)           -> logits   (standard classification)
      - extract_features(x)  -> embeddings (for metric learning / DML)
      - get_embedding_dim()  -> int       (embedding dimensionality)

    These interfaces allow the backbone to be swapped to a Vision
    Transformer or used with Proxy Anchor Loss without changing the
    training pipeline.
    """
    def __init__(self, num_classes=2, embedding_dim=128, dropout=0.5, block_dropout=0.25):
        super().__init__()

        # Feature extraction blocks
        self.block1 = ConvBlock(3, 32, dropout_rate=block_dropout)
        self.block2 = ConvBlock(32, 64, dropout_rate=block_dropout)
        self.block3 = ConvBlock(64, 128, dropout_rate=block_dropout)

        # Final convolution block (without max pool — uses GAP instead)
        self.final_conv = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Embedding layer (for future metric learning compatibility)
        self.embedding = nn.Sequential(
            nn.Linear(256, embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )

        # Classification head
        self.classifier = nn.Linear(embedding_dim, num_classes)
        self._embedding_dim = embedding_dim
        self._num_classes = num_classes

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Kaiming initialization for conv layers, constant for BN."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def extract_features(self, x):
        """
        Extract feature embeddings from the input image.
        Returns: (B, embedding_dim) tensor

        This method is the primary interface for Deep Metric Learning
        and Proxy Anchor Loss integration.
        """
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.final_conv(x)
        x = self.gap(x)
        x = x.flatten(1)
        x = self.embedding(x)
        return x

    def forward(self, x):
        """Standard forward pass returning class logits."""
        features = self.extract_features(x)
        logits = self.classifier(features)
        return logits

    def get_embedding_dim(self):
        """Return the dimensionality of the embedding layer."""
        return self._embedding_dim


print("\u2713 OMNet-V1 architecture defined")
print("  Interfaces: forward(), extract_features(), get_embedding_dim()")

# Section 11: Model Summary

**Purpose:** Instantiate the model, print its architecture, count parameters, and verify forward pass shapes.

**Theory:** Before training, it is essential to verify that the model architecture is correct by checking output shapes and parameter counts. This catches dimension mismatches and ensures the computational graph is valid.

**Expected Output:** Model architecture, parameter table, forward pass shape verification.

In [ ]:
# ============================================================
# Section 11: Model Summary
# ============================================================

# Instantiate model
model = OMNetV1(
    num_classes=CONFIG['model']['num_classes'],
    embedding_dim=CONFIG['model']['embedding_dim'],
    dropout=CONFIG['model']['dropout'],
    block_dropout=CONFIG['model']['block_dropout'],
).to(DEVICE)

# --- Parameter Count ---
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("  OMNet-V1 Model Summary")
print("=" * 60)
print(model)
print("\n" + "-" * 60)

# Per-layer parameter table
print(f"\n{'Layer':<40s} {'Params':>12s}")
print("-" * 54)
for name, param in model.named_parameters():
    print(f"  {name:<38s} {param.numel():>10,d}")
print("-" * 54)
print(f"  {'Total':<38s} {total_params:>10,d}")
print(f"  {'Trainable':<38s} {trainable_params:>10,d}")
print(f"  {'Non-trainable':<38s} {total_params - trainable_params:>10,d}")

# --- Forward Pass Verification ---
print("\n" + "-" * 60)
print("  Forward Pass Verification")
print("-" * 60)
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

with torch.no_grad():
    logits = model(dummy_input)
    embeddings = model.extract_features(dummy_input)

print(f"  Input shape       : {tuple(dummy_input.shape)}")
print(f"  Logits shape      : {tuple(logits.shape)}   (expected: (1, {CONFIG['model']['num_classes']}))")
print(f"  Embedding shape   : {tuple(embeddings.shape)}   (expected: (1, {CONFIG['model']['embedding_dim']}))")
print(f"  Embedding dim     : {model.get_embedding_dim()}")

assert logits.shape == (1, CONFIG['model']['num_classes']), "Logits shape mismatch!"
assert embeddings.shape == (1, CONFIG['model']['embedding_dim']), "Embedding shape mismatch!"
print("\n  \u2705 All shape assertions passed!")
print("=" * 60)

del dummy_input, logits, embeddings
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

# Section 12: Training Loop

**Purpose:** Define loss functions and the `Trainer` class that encapsulates the full training pipeline with validation, logging, checkpointing, and early stopping.

**Theory:**
- **CrossEntropyLoss** with class weights handles the 2:1 class imbalance.
- **Focal Loss** (optional) down-weights well-classified examples, focusing learning on hard cases: `FL(p) = -α(1-p)^γ log(p)`.
- **Proxy Anchor Loss** is defined as a placeholder interface for future metric learning experiments.
- **Mixed precision** (FP16) halves memory usage and accelerates training on Tensor Cores.
- **Early stopping** prevents overfitting by monitoring validation AUC.

**Expected Output:** Loss function and Trainer class definitions (training runs in Section 13).

In [ ]:
# ============================================================
# Section 12: Training Loop — Loss Functions & Trainer
# ============================================================

# -------------------------
# Loss Functions
# -------------------------

class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Args:
        alpha: Class weights tensor (auto-computed if None)
        gamma: Focusing parameter (default 2.0)
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class ProxyAnchorLoss(nn.Module):
    """
    Proxy Anchor Loss for Deep Metric Learning (Kim et al., CVPR 2020).

    PLACEHOLDER: This loss is defined but NOT connected to the training
    loop. It will be used with model.extract_features() in future
    experiments with Deep Metric Learning.

    Args:
        nb_classes: Number of classes
        sz_embed: Embedding dimensionality (must match model.get_embedding_dim())
        mrg: Margin (default 0.1)
        alpha: Scaling factor (default 32)
    """
    def __init__(self, nb_classes, sz_embed, mrg=0.1, alpha=32):
        super().__init__()
        self.proxies = nn.Parameter(torch.randn(nb_classes, sz_embed))
        nn.init.kaiming_normal_(self.proxies, mode='fan_out')
        self.nb_classes = nb_classes
        self.sz_embed = sz_embed
        self.mrg = mrg
        self.alpha = alpha

    def forward(self, embeddings, labels):
        """Compute Proxy Anchor Loss between embeddings and proxies."""
        X = F.normalize(embeddings, p=2, dim=-1)
        P = F.normalize(self.proxies, p=2, dim=-1)
        cos = F.linear(X, P)
        P_one_hot = F.one_hot(labels, self.nb_classes).float()
        N_one_hot = 1 - P_one_hot
        pos_exp = torch.exp(-self.alpha * (cos - self.mrg))
        neg_exp = torch.exp(self.alpha * (cos + self.mrg))
        with_pos_proxies = torch.nonzero(P_one_hot.sum(dim=0) != 0).squeeze(1)
        num_valid = max(len(with_pos_proxies), 1)
        pos_term = torch.log(1 + (P_one_hot * pos_exp).sum(dim=0)).sum() / num_valid
        neg_term = torch.log(1 + (N_one_hot * neg_exp).sum(dim=0)).sum() / self.nb_classes
        return pos_term + neg_term


def build_criterion(config, class_weights, device):
    """Factory function to build the loss criterion based on config."""
    loss_name = config['loss']['name']
    if loss_name == 'focal':
        alpha = class_weights if config['loss']['focal_alpha'] is None else                 torch.tensor(config['loss']['focal_alpha'], dtype=torch.float32).to(device)
        criterion = FocalLoss(alpha=alpha, gamma=config['loss']['focal_gamma'])
        print(f"  Loss: Focal Loss (gamma={config['loss']['focal_gamma']})")
    else:
        criterion = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=config['loss']['label_smoothing'],
        )
        print(f"  Loss: CrossEntropy (smoothing={config['loss']['label_smoothing']})")
    return criterion


# -------------------------
# Trainer Class
# -------------------------

class Trainer:
    """
    Training engine for OMNet.

    Handles the full training loop including:
    - Per-epoch training and validation
    - Mixed precision (AMP)
    - Early stopping (by validation AUC)
    - Checkpoint saving (best + last)
    - Comprehensive per-epoch logging
    """
    def __init__(self, model, train_loader, val_loader, criterion,
                 optimizer, scheduler, config, device, output_dir):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.config = config
        self.device = device
        self.output_dir = Path(output_dir)
        self.history = []
        self.best_val_auc = 0.0
        self.best_epoch = 0
        self.patience_counter = 0

        # Mixed precision
        use_amp = config['training']['mixed_precision'] and device.type == 'cuda'
        self.scaler = torch.amp.GradScaler() if use_amp else None
        self.use_amp = use_amp

    def _compute_metrics(self, y_true, y_pred, y_prob):
        """Compute all classification metrics."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        y_prob = np.array(y_prob)

        # Confusion matrix components
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = 0.0
        try:
            pr_auc = average_precision_score(y_true, y_prob)
        except ValueError:
            pr_auc = 0.0

        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'specificity': specificity,
            'sensitivity': sensitivity,
            'roc_auc': auc,
            'pr_auc': pr_auc,
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'mcc': matthews_corrcoef(y_true, y_pred),
        }

    def train_one_epoch(self):
        """Run one epoch of training."""
        self.model.train()
        running_loss = 0.0
        all_preds, all_labels, all_probs = [], [], []

        for images, labels in self.train_loader:
            images, labels = images.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()

            if self.use_amp:
                with torch.amp.autocast(device_type='cuda'):
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs.detach(), dim=1)
            all_preds.extend(probs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

        epoch_loss = running_loss / len(self.train_loader.dataset)
        metrics = self._compute_metrics(all_labels, all_preds, all_probs)
        metrics['loss'] = epoch_loss
        return metrics

    @torch.no_grad()
    def validate(self):
        """Run validation."""
        self.model.eval()
        running_loss = 0.0
        all_preds, all_labels, all_probs = [], [], []

        for images, labels in self.val_loader:
            images, labels = images.to(self.device), labels.to(self.device)

            if self.use_amp:
                with torch.amp.autocast(device_type='cuda'):
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            all_preds.extend(probs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

        epoch_loss = running_loss / len(self.val_loader.dataset)
        metrics = self._compute_metrics(all_labels, all_preds, all_probs)
        metrics['loss'] = epoch_loss
        return metrics

    def _save_checkpoint(self, epoch, metrics, filename):
        """Save model checkpoint."""
        path = self.output_dir / filename
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
            'metrics': metrics,
            'config': self.config,
        }, path)

    def fit(self):
        """Run the full training loop."""
        num_epochs = 2 if self.config.get('smoke_test') else self.config['training']['epochs']
        patience = self.config['training']['early_stopping_patience']
        start_time = time.time()

        print("\n" + "=" * 70)
        print(f"  Training OMNet-V1 | Epochs: {num_epochs} | Device: {self.device}")
        print(f"  Mixed Precision: {self.use_amp} | Early Stopping Patience: {patience}")
        print("=" * 70)

        for epoch in range(num_epochs):
            epoch_start = time.time()

            # --- Train ---
            train_metrics = self.train_one_epoch()

            # --- Validate ---
            val_metrics = self.validate()

            # --- Scheduler Step ---
            if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_metrics['loss'])
            elif self.scheduler is not None:
                self.scheduler.step()

            current_lr = self.optimizer.param_groups[0]['lr']
            epoch_time = time.time() - epoch_start
            elapsed = time.time() - start_time
            eta = (elapsed / (epoch + 1)) * (num_epochs - epoch - 1)

            # GPU memory usage
            gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

            # --- Build epoch record ---
            record = {
                'epoch': epoch + 1,
                'train_loss': round(train_metrics['loss'], 6),
                'val_loss': round(val_metrics['loss'], 6),
                'train_acc': round(train_metrics['accuracy'], 4),
                'val_acc': round(val_metrics['accuracy'], 4),
                'val_precision': round(val_metrics['precision'], 4),
                'val_recall': round(val_metrics['recall'], 4),
                'val_f1': round(val_metrics['f1'], 4),
                'val_roc_auc': round(val_metrics['roc_auc'], 4),
                'val_pr_auc': round(val_metrics['pr_auc'], 4),
                'val_specificity': round(val_metrics['specificity'], 4),
                'val_sensitivity': round(val_metrics['sensitivity'], 4),
                'val_balanced_acc': round(val_metrics['balanced_accuracy'], 4),
                'val_mcc': round(val_metrics['mcc'], 4),
                'lr': current_lr,
                'gpu_mem_gb': round(gpu_mem, 3),
                'epoch_time_s': round(epoch_time, 1),
            }
            self.history.append(record)

            # --- Print epoch summary ---
            eta_str = str(timedelta(seconds=int(eta)))
            print(f"\nEpoch {epoch+1}/{num_epochs} | Time: {epoch_time:.1f}s | ETA: {eta_str}")
            print(f"  Train \u2014 Loss: {train_metrics['loss']:.4f}  Acc: {train_metrics['accuracy']:.4f}  AUC: {train_metrics['roc_auc']:.4f}")
            print(f"  Val   \u2014 Loss: {val_metrics['loss']:.4f}  Acc: {val_metrics['accuracy']:.4f}  AUC: {val_metrics['roc_auc']:.4f}")
            print(f"  Val   \u2014 Prec: {val_metrics['precision']:.4f}  Rec: {val_metrics['recall']:.4f}  F1: {val_metrics['f1']:.4f}")
            print(f"  LR: {current_lr:.6f} | GPU Mem: {gpu_mem:.2f} GB")

            # --- Save last model ---
            self._save_checkpoint(epoch, val_metrics, 'last_model.pth')

            # --- Check for best model ---
            if val_metrics['roc_auc'] > self.best_val_auc:
                self.best_val_auc = val_metrics['roc_auc']
                self.best_epoch = epoch + 1
                self.patience_counter = 0
                self._save_checkpoint(epoch, val_metrics, 'best_model.pth')
                print(f"  \u2605 New best model (AUC: {self.best_val_auc:.4f})")
            else:
                self.patience_counter += 1
                if self.patience_counter >= patience and not self.config.get('smoke_test'):
                    print(f"\n\u23f9 Early stopping at epoch {epoch+1} (patience: {patience})")
                    break

        total_time = time.time() - start_time
        print("\n" + "=" * 70)
        print(f"  Training Complete | Total: {timedelta(seconds=int(total_time))}")
        print(f"  Best epoch: {self.best_epoch} | Best AUC: {self.best_val_auc:.4f}")
        print("=" * 70)
        return self.history


print("\u2713 Loss functions defined: CrossEntropyLoss, FocalLoss, ProxyAnchorLoss (placeholder)")
print("\u2713 Trainer class defined")

# Section 13: Validation — Run Training

**Purpose:** Initialize the optimizer, scheduler, and loss function, then run the training loop. Validation is performed automatically after each epoch inside `trainer.fit()`.

**Theory:** We use the Adam optimizer with weight decay for stable convergence. CosineAnnealingLR gradually reduces the learning rate following a cosine schedule, which has been shown to improve final performance compared to fixed or step-decay schedules.

**Expected Output:** Per-epoch training/validation metrics, best model checkpoint saved.

In [ ]:
# ============================================================
# Section 13: Run Training (with per-epoch Validation)
# ============================================================

# --- Build loss function ---
print("Initializing training pipeline...\n")
criterion = build_criterion(CONFIG, class_weights, DEVICE)

# --- Build optimizer ---
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['training']['lr'],
    weight_decay=CONFIG['training']['weight_decay'],
)
print(f"  Optimizer: Adam (lr={CONFIG['training']['lr']}, wd={CONFIG['training']['weight_decay']})")

# --- Build scheduler ---
num_epochs = 2 if CONFIG.get('smoke_test') else CONFIG['training']['epochs']
if CONFIG['training']['scheduler'] == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    print(f"  Scheduler: CosineAnnealingLR (T_max={num_epochs})")
else:
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    print(f"  Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")

# --- Create Trainer & Run ---
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=CONFIG,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

training_history = trainer.fit()

# Convert to DataFrame for analysis
history_df = pd.DataFrame(training_history)
print("\nTraining history DataFrame:")
print(history_df.to_string(index=False))

# Section 14: Evaluation

**Purpose:** Load the best model checkpoint and perform comprehensive evaluation on the held-out test set.

**Theory:** Test set evaluation provides an unbiased estimate of model generalization. We compute 13 metrics covering accuracy, precision, recall, specificity, sensitivity, F1, ROC-AUC, PR-AUC, balanced accuracy, and Matthews Correlation Coefficient. MCC is particularly valuable for imbalanced datasets as it considers all four confusion matrix quadrants.

**Expected Output:** Full test metrics table, classification report, prediction distribution.

In [ ]:
# ============================================================
# Section 14: Evaluation on Test Set
# ============================================================

# Load best model checkpoint
best_ckpt_path = OUTPUT_DIR / 'best_model.pth'
if best_ckpt_path.exists():
    checkpoint = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\u2713 Loaded best model from epoch {checkpoint['epoch'] + 1}")
else:
    print("\u26a0\ufe0f  No best checkpoint found. Using current model weights.")

model.eval()

# --- Run inference on test set ---
test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Evaluating'):
        images = images.to(DEVICE)
        if trainer.use_amp:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
        else:
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        test_preds.extend(probs.argmax(1).cpu().numpy())
        test_labels.extend(labels.numpy())
        test_probs.extend(probs[:, 1].cpu().numpy())

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)
test_probs = np.array(test_probs)

# --- Compute all metrics ---
cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

test_metrics = {
    'Accuracy': accuracy_score(test_labels, test_preds),
    'Precision': precision_score(test_labels, test_preds, zero_division=0),
    'Recall': recall_score(test_labels, test_preds, zero_division=0),
    'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
    'F1-Score': f1_score(test_labels, test_preds, zero_division=0),
    'ROC-AUC': roc_auc_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0,
    'PR-AUC': average_precision_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0,
    'Balanced Accuracy': balanced_accuracy_score(test_labels, test_preds),
    'MCC': matthews_corrcoef(test_labels, test_preds),
}

# --- Print Results ---
print("\n" + "=" * 60)
print("  Test Set Evaluation Results")
print("=" * 60)
for metric_name, value in test_metrics.items():
    print(f"  {metric_name:<22s}: {value:.4f}")
print("=" * 60)

# --- Classification Report ---
print("\nClassification Report:")
print(classification_report(test_labels, test_preds,
                            target_names=CLASS_NAMES, digits=4))

# --- Prediction Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of predicted probabilities
axes[0].hist(test_probs[test_labels == 0], bins=30, alpha=0.7, label='Benign', color='#2ecc71', edgecolor='white')
axes[0].hist(test_probs[test_labels == 1], bins=30, alpha=0.7, label='Malignant', color='#e74c3c', edgecolor='white')
axes[0].set_xlabel('Predicted Probability (Malignant)')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Distribution by True Class', fontweight='bold')
axes[0].legend()
axes[0].axvline(0.5, color='black', linestyle='--', alpha=0.5, label='Threshold')

# Pie chart of predictions
pred_counts = Counter(test_preds)
axes[1].pie([pred_counts.get(0, 0), pred_counts.get(1, 0)],
            labels=['Benign', 'Malignant'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Prediction Distribution', fontweight='bold')

plt.tight_layout()
save_figure(fig, 'prediction_distribution.png')
plt.show()

# Section 15: Error Analysis

**Purpose:** Identify and analyze misclassified test samples to understand model failure patterns.

**Theory:** Error analysis reveals systematic weaknesses. Common failure patterns in histopathology include: borderline cases with mixed benign/malignant features, unusual staining artifacts, or rare tissue morphologies. Understanding these patterns guides architecture improvements and data augmentation strategies.

**Expected Output:** Grid of misclassified images with true/predicted labels and confidence scores, confidence histogram.

In [ ]:
# ============================================================
# Section 15: Error Analysis
# ============================================================

# Identify misclassified samples
misclassified_mask = test_preds != test_labels
misclassified_indices = np.where(misclassified_mask)[0]
correct_mask = ~misclassified_mask
correct_indices = np.where(correct_mask)[0]

print(f"Total test samples: {len(test_labels)}")
print(f"Correct: {correct_mask.sum()} ({correct_mask.mean()*100:.1f}%)")
print(f"Misclassified: {misclassified_mask.sum()} ({misclassified_mask.mean()*100:.1f}%)")

# Confusion breakdown
fn_indices = np.where((test_labels == 1) & (test_preds == 0))[0]
fp_indices = np.where((test_labels == 0) & (test_preds == 1))[0]
print(f"  False Negatives (Malignant predicted as Benign): {len(fn_indices)}")
print(f"  False Positives (Benign predicted as Malignant): {len(fp_indices)}")

# --- Misclassified Images Grid ---
n_show = min(16, len(misclassified_indices))
if n_show > 0:
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    fig.suptitle('Misclassified Test Images', fontsize=16, fontweight='bold', y=1.02)

    for i in range(16):
        row, col = divmod(i, 4)
        if i < n_show:
            idx = misclassified_indices[i]
            img = Image.open(test_dataset.samples[idx]).convert('RGB')
            axes[row, col].imshow(img)
            true_label = CLASS_NAMES[test_labels[idx]]
            pred_label = CLASS_NAMES[test_preds[idx]]
            conf = test_probs[idx] if test_preds[idx] == 1 else 1 - test_probs[idx]
            axes[row, col].set_title(
                f'True: {true_label}\nPred: {pred_label} ({conf:.2f})',
                fontsize=9, color='red'
            )
            # Save misclassified image
            img.save(MISCLASSIFIED_DIR / f'misclassified_{i+1}_{true_label}_as_{pred_label}.png')
        axes[row, col].axis('off')

    plt.tight_layout()
    save_figure(fig, 'misclassified_samples.png')
    plt.show()
else:
    print("No misclassified samples!")

# --- Confidence Histogram: Correct vs Incorrect ---
fig, ax = plt.subplots(figsize=(10, 6))

# Confidence = probability assigned to the predicted class
confidences = np.where(test_preds == 1, test_probs, 1 - test_probs)
correct_conf = confidences[correct_mask]
incorrect_conf = confidences[misclassified_mask]

ax.hist(correct_conf, bins=30, alpha=0.7, label=f'Correct (n={len(correct_conf)})',
        color='#2ecc71', edgecolor='white')
if len(incorrect_conf) > 0:
    ax.hist(incorrect_conf, bins=30, alpha=0.7, label=f'Incorrect (n={len(incorrect_conf)})',
            color='#e74c3c', edgecolor='white')
ax.set_xlabel('Prediction Confidence')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution: Correct vs. Incorrect Predictions', fontweight='bold')
ax.legend()

plt.tight_layout()
save_figure(fig, 'confidence_histogram.png')
plt.show()

# --- Save Correct Predictions (top 16 by confidence) ---
if len(correct_indices) > 0:
    correct_conf_vals = confidences[correct_indices]
    top_correct = correct_indices[np.argsort(-correct_conf_vals)][:16]

    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    fig.suptitle('Top Correct Predictions (Highest Confidence)', fontsize=16, fontweight='bold', y=1.02)
    for i in range(16):
        row, col = divmod(i, 4)
        if i < len(top_correct):
            idx = top_correct[i]
            img = Image.open(test_dataset.samples[idx]).convert('RGB')
            axes[row, col].imshow(img)
            label = CLASS_NAMES[test_labels[idx]]
            conf = confidences[idx]
            axes[row, col].set_title(f'{label} ({conf:.3f})', fontsize=10, color='green')
            img.save(CORRECT_PRED_DIR / f'correct_{i+1}_{label}.png')
        axes[row, col].axis('off')
    plt.tight_layout()
    save_figure(fig, 'correct_predictions.png')
    plt.show()

# Section 16: Grad-CAM

**Purpose:** Generate Gradient-weighted Class Activation Maps (Grad-CAM) to visualize which image regions the model focuses on for its predictions.

**Theory:** Grad-CAM uses the gradients of the target class flowing into the final convolutional layer to produce a coarse localization map highlighting important regions. For histopathology, this reveals whether the model attends to pathologically relevant features (e.g., cell clusters, tissue boundaries) or spurious artifacts.

**Expected Output:** Heatmap overlays on correctly classified and misclassified samples.

In [ ]:
# ============================================================
# Section 16: Grad-CAM Visualization
# ============================================================
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target the last convolutional layer in the network
# final_conv[3] is the second Conv2d(256, 256)
target_layers = [model.final_conv[3]]

cam = GradCAM(model=model, target_layers=target_layers)

def generate_gradcam_grid(indices, title, filename, max_show=8):
    """Generate a Grad-CAM visualization grid for given sample indices."""
    n_show = min(max_show, len(indices))
    if n_show == 0:
        print(f"  No samples available for: {title}")
        return

    n_cols = 4
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, 2 * n_cols, figsize=(24, 6 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)

    for i in range(n_show):
        idx = indices[i]
        row = i // n_cols
        col_base = (i % n_cols) * 2

        # Load and prepare image
        img_pil = Image.open(test_dataset.samples[idx]).convert('RGB')
        img_tensor = eval_transform(img_pil).unsqueeze(0).to(DEVICE)

        # Original image (normalized to [0,1] for overlay)
        img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
        img_np = np.array(img_resized).astype(np.float32) / 255.0

        # Generate CAM
        pred_class = int(test_preds[idx])
        targets = [ClassifierOutputTarget(pred_class)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]

        # Overlay
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        # Plot original
        axes[row, col_base].imshow(img_np)
        true_label = CLASS_NAMES[test_labels[idx]]
        pred_label = CLASS_NAMES[test_preds[idx]]
        axes[row, col_base].set_title(f'True: {true_label}', fontsize=10)
        axes[row, col_base].axis('off')

        # Plot Grad-CAM
        axes[row, col_base + 1].imshow(cam_image)
        axes[row, col_base + 1].set_title(f'Pred: {pred_label}', fontsize=10)
        axes[row, col_base + 1].axis('off')

        # Save individual Grad-CAM
        cam_pil = Image.fromarray(cam_image)
        cam_pil.save(GRADCAM_DIR / f'gradcam_{filename}_{i+1}.png')

    # Hide unused axes
    for i in range(n_show, n_rows * n_cols):
        row = i // n_cols
        col_base = (i % n_cols) * 2
        if row < axes.shape[0] and col_base + 1 < axes.shape[1]:
            axes[row, col_base].axis('off')
            axes[row, col_base + 1].axis('off')

    plt.tight_layout()
    save_figure(fig, f'gradcam_{filename}.png')
    plt.show()

# Generate Grad-CAMs for correct predictions (4 benign + 4 malignant)
correct_benign = [i for i in correct_indices if test_labels[i] == 0][:4]
correct_malig = [i for i in correct_indices if test_labels[i] == 1][:4]
correct_combined = list(correct_benign) + list(correct_malig)

generate_gradcam_grid(correct_combined, 'Grad-CAM: Correctly Classified Samples', 'correct')

# Generate Grad-CAMs for misclassified predictions
generate_gradcam_grid(misclassified_indices[:8], 'Grad-CAM: Misclassified Samples', 'misclassified')

del cam
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
print("\n\u2713 Grad-CAM visualizations saved to:", GRADCAM_DIR)

# Section 17: ROC Curve

**Purpose:** Plot the Receiver Operating Characteristic (ROC) curve for the test set.

**Theory:** The ROC curve plots True Positive Rate (Sensitivity) vs. False Positive Rate (1 - Specificity) at all classification thresholds. The Area Under the Curve (AUC) summarizes discriminative performance: AUC = 1.0 is perfect, AUC = 0.5 is random chance.

**Expected Output:** ROC curve with AUC annotation.

In [ ]:
# ============================================================
# Section 17: ROC Curve
# ============================================================
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
roc_auc = roc_auc_score(test_labels, test_probs)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='#e74c3c', linewidth=2.5, label=f'OMNet-V1 (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5, label='Random Chance')
ax.fill_between(fpr, tpr, alpha=0.15, color='#e74c3c')

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=13)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=13)
ax.set_title('ROC Curve \u2014 Test Set', fontsize=15, fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_aspect('equal')

plt.tight_layout()
save_figure(fig, 'roc_curve.png')
plt.show()
print(f"ROC-AUC: {roc_auc:.4f}")

# Section 18: Precision-Recall Curve

**Purpose:** Plot the Precision-Recall curve for the test set.

**Theory:** The PR curve is especially informative for imbalanced datasets. It plots Precision (positive predictive value) against Recall (sensitivity) at all thresholds. PR-AUC summarizes performance: higher is better. Unlike ROC-AUC, PR-AUC is not inflated by a large number of true negatives.

**Expected Output:** PR curve with AP (Average Precision) annotation.

In [ ]:
# ============================================================
# Section 18: Precision-Recall Curve
# ============================================================
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(test_labels, test_probs)
pr_auc = average_precision_score(test_labels, test_probs)

# Baseline: proportion of positive class
baseline = test_labels.mean()

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(recall_vals, precision_vals, color='#3498db', linewidth=2.5,
        label=f'OMNet-V1 (AP = {pr_auc:.4f})')
ax.axhline(y=baseline, color='gray', linestyle='--', linewidth=1.5,
           label=f'Random Baseline ({baseline:.3f})')
ax.fill_between(recall_vals, precision_vals, alpha=0.15, color='#3498db')

ax.set_xlabel('Recall (Sensitivity)', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curve \u2014 Test Set', fontsize=15, fontweight='bold')
ax.legend(loc='lower left', fontsize=12)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([0, 1.05])

plt.tight_layout()
save_figure(fig, 'precision_recall.png')
plt.show()
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")

# Section 19: Confusion Matrix

**Purpose:** Visualize the confusion matrix for the test set in both absolute and normalized forms.

**Theory:** The confusion matrix shows the counts of true positives, true negatives, false positives, and false negatives. The normalized version reveals per-class accuracy rates. For medical diagnosis, false negatives (missing malignant cases) are typically more costly than false positives.

**Expected Output:** Side-by-side confusion matrices (absolute + normalized).

In [ ]:
# ============================================================
# Section 19: Confusion Matrix
# ============================================================
cm_raw = confusion_matrix(test_labels, test_preds, labels=[0, 1])
cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Absolute counts
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={'size': 16}, linewidths=1, linecolor='white')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Reds', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={'size': 16}, linewidths=1, linecolor='white',
            vmin=0, vmax=1)
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')

plt.tight_layout()
save_figure(fig, 'confusion_matrix.png')
plt.show()

print(f"\nTN={cm_raw[0,0]}  FP={cm_raw[0,1]}")
print(f"FN={cm_raw[1,0]}  TP={cm_raw[1,1]}")

# Section 20: Training Curves

**Purpose:** Visualize training dynamics over epochs: loss, accuracy, learning rate, and ROC-AUC.

**Theory:** Training curves reveal convergence behavior, overfitting (diverging train/val curves), and the effect of learning rate scheduling. Healthy training shows decreasing loss, increasing accuracy, and a gap between train and validation metrics that doesn't widen significantly.

**Expected Output:** 2×2 subplot grid showing Loss, Accuracy, LR schedule, and AUC over epochs.

In [ ]:
# ============================================================
# Section 20: Training Curves
# ============================================================
epochs_range = history_df['epoch'].values

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Training Curves', fontsize=16, fontweight='bold', y=1.02)

# --- Loss ---
axes[0, 0].plot(epochs_range, history_df['train_loss'], 'o-', label='Train Loss', color='#e74c3c', markersize=4)
axes[0, 0].plot(epochs_range, history_df['val_loss'], 's-', label='Val Loss', color='#3498db', markersize=4)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Loss', fontweight='bold')
axes[0, 0].legend()

# --- Accuracy ---
axes[0, 1].plot(epochs_range, history_df['train_acc'], 'o-', label='Train Acc', color='#e74c3c', markersize=4)
axes[0, 1].plot(epochs_range, history_df['val_acc'], 's-', label='Val Acc', color='#3498db', markersize=4)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy', fontweight='bold')
axes[0, 1].legend()

# --- Learning Rate ---
axes[1, 0].plot(epochs_range, history_df['lr'], 'D-', color='#9b59b6', markersize=4)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 0].ticklabel_format(style='scientific', axis='y', scilimits=(0, 0))

# --- ROC-AUC ---
axes[1, 1].plot(epochs_range, history_df['val_roc_auc'], '^-', label='Val AUC', color='#2ecc71', markersize=4)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('ROC-AUC')
axes[1, 1].set_title('Validation ROC-AUC', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
save_figure(fig, 'training_curve.png')
plt.show()

# Feature Embedding Visualization (t-SNE)

**Purpose:** Visualize the learned feature space using t-SNE dimensionality reduction.

**Theory:** t-SNE (t-distributed Stochastic Neighbor Embedding) projects high-dimensional embeddings into 2D while preserving local neighborhood structure. Well-separated clusters in the embedding space indicate that the model has learned discriminative features for each class. This visualization also validates that `extract_features()` produces meaningful representations for future metric learning experiments.

**Expected Output:** 2D scatter plot of test set embeddings colored by true class label.

In [ ]:
# ============================================================
# Feature Embedding Visualization (t-SNE)
# ============================================================
print("Extracting features from test set...")

@torch.no_grad()
def extract_all_features(model, dataloader, device):
    """Extract embeddings from all samples in a dataloader."""
    model.eval()
    all_features = []
    all_labels = []
    for images, labels in tqdm(dataloader, desc='Extracting features'):
        images = images.to(device)
        if trainer.use_amp:
            with torch.amp.autocast(device_type='cuda'):
                features = model.extract_features(images)
        else:
            features = model.extract_features(images)
        all_features.append(features.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.concatenate(all_features), np.array(all_labels)

features_np, labels_np = extract_all_features(model, test_loader, DEVICE)

print(f"Feature matrix shape: {features_np.shape}")
print("Running t-SNE (this may take a moment)...")

tsne = TSNE(n_components=2, perplexity=min(30, len(features_np) - 1),
            random_state=CONFIG['seed'], n_iter=1000)
embeddings_2d = tsne.fit_transform(features_np)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2ecc71', '#e74c3c']

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = labels_np == cls_idx
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
              c=colors[cls_idx], label=cls_name.capitalize(),
              alpha=0.6, s=30, edgecolors='white', linewidths=0.5)

ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('t-SNE Feature Embedding Visualization (Test Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, markerscale=2)

plt.tight_layout()
save_figure(fig, 'tsne_embeddings.png')
plt.show()

del features_np, embeddings_2d
gc.collect()

# Section 21: Save Best Model

**Purpose:** Verify saved checkpoints and create a clean inference-only model file.

**Theory:** Saving both the full checkpoint (for resuming training) and a lightweight state dict (for inference) provides flexibility. The full checkpoint includes optimizer state, scheduler state, and metrics for reproducibility.

**Expected Output:** Verification of saved model files, file sizes.

In [ ]:
# ============================================================
# Section 21: Save Best Model
# ============================================================
print("=" * 60)
print("  Model Checkpoint Verification")
print("=" * 60)

model_files = ['best_model.pth', 'last_model.pth']
for fname in model_files:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        ckpt = torch.load(fpath, map_location='cpu', weights_only=False)
        print(f"  \u2713 {fname}")
        print(f"      Size  : {size_mb:.2f} MB")
        print(f"      Epoch : {ckpt.get('epoch', 'N/A')}")
        if 'metrics' in ckpt:
            print(f"      AUC   : {ckpt['metrics'].get('roc_auc', 'N/A')}")
    else:
        print(f"  \u274c {fname} \u2014 NOT FOUND")

# Verify checkpoint round-trip
print("\n  Verifying checkpoint round-trip...")
test_model = OMNetV1(
    num_classes=CONFIG['model']['num_classes'],
    embedding_dim=CONFIG['model']['embedding_dim'],
    dropout=CONFIG['model']['dropout'],
    block_dropout=CONFIG['model']['block_dropout'],
).to(DEVICE)

ckpt = torch.load(OUTPUT_DIR / 'best_model.pth', map_location=DEVICE, weights_only=False)
test_model.load_state_dict(ckpt['model_state_dict'])
test_model.eval()

with torch.no_grad():
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out = test_model(dummy)
    assert out.shape == (1, CONFIG['model']['num_classes'])
print("  \u2705 Checkpoint round-trip verified!")

del test_model, ckpt
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
print("=" * 60)

# Section 22: Export Metrics

**Purpose:** Export training history, test metrics, and experiment configuration to CSV and JSON files for analysis and comparison with future experiments.

**Theory:** Structured exports enable systematic comparison across experiments (OMNet-V1 vs. V2 vs. ViT vs. Proxy Anchor Loss). The CSV format is compatible with pandas and spreadsheet tools. The JSON file captures the full experiment context for reproducibility.

**Expected Output:** `training_history.csv`, `metrics.csv`, `experiment_summary.json` saved to outputs/.

In [ ]:
# ============================================================
# Section 22: Export Metrics
# ============================================================

# --- Training History CSV ---
history_df.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
print(f"\u2713 Saved: {OUTPUT_DIR / 'training_history.csv'}")

# --- Test Metrics CSV ---
metrics_df = pd.DataFrame([test_metrics])
metrics_df.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
print(f"\u2713 Saved: {OUTPUT_DIR / 'metrics.csv'}")

# --- Experiment Summary JSON ---
experiment_summary = {
    'experiment': {
        'name': 'OMNet-V1 Baseline',
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'smoke_test': CONFIG.get('smoke_test', False),
    },
    'config': CONFIG,
    'dataset': {
        'name': 'BreakHis 400X',
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
        'test_size': len(test_dataset),
        'normalization_mean': NORM_MEAN,
        'normalization_std': NORM_STD,
    },
    'model': {
        'name': 'OMNet-V1',
        'total_params': sum(p.numel() for p in model.parameters()),
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'embedding_dim': model.get_embedding_dim(),
    },
    'training': {
        'total_epochs': len(training_history),
        'best_epoch': trainer.best_epoch,
        'best_val_auc': trainer.best_val_auc,
        'total_time_s': sum(r['epoch_time_s'] for r in training_history),
    },
    'test_metrics': {k: round(v, 6) for k, v in test_metrics.items()},
}

with open(OUTPUT_DIR / 'experiment_summary.json', 'w') as f:
    json.dump(experiment_summary, f, indent=2)
print(f"\u2713 Saved: {OUTPUT_DIR / 'experiment_summary.json'}")

# --- List all output files ---
print("\n" + "=" * 60)
print("  All Output Files")
print("=" * 60)
for item in sorted(OUTPUT_DIR.rglob('*')):
    if item.is_file():
        size_kb = item.stat().st_size / 1024
        rel_path = item.relative_to(OUTPUT_DIR)
        print(f"  {rel_path}  ({size_kb:.1f} KB)")
print("=" * 60)

# Section 23: Final Experiment Summary

**Purpose:** Print a comprehensive summary card of the entire experiment for quick reference and documentation.

**Interpretation:** This summary captures the full experiment context. Compare these results with future experiments (OMNet-V2, Vision Transformer, Proxy Anchor Loss) to evaluate architectural improvements. The modular interfaces (`extract_features()`, `get_embedding_dim()`) ensure that only the backbone and loss function need to change in future versions — the training pipeline remains untouched.

**Expected Output:** Formatted experiment summary card.

In [ ]:
# ============================================================
# Section 23: Final Experiment Summary
# ============================================================
total_time = sum(r['epoch_time_s'] for r in training_history)
total_params = sum(p.numel() for p in model.parameters())

print()
print("\u2554" + "\u2550" * 62 + "\u2557")
print("\u2551" + "  \U0001f52c OMNet-V1 Experiment Summary".ljust(62) + "\u2551")
print("\u2560" + "\u2550" * 62 + "\u2563")
print("\u2551" + f"  Model          : OMNet-V1 (Custom CNN)".ljust(62) + "\u2551")
print("\u2551" + f"  Parameters     : {total_params:,d}".ljust(62) + "\u2551")
print("\u2551" + f"  Embedding Dim  : {model.get_embedding_dim()}".ljust(62) + "\u2551")
print("\u2551" + f"  Epochs Trained : {len(training_history)}".ljust(62) + "\u2551")
print("\u2551" + f"  Best Epoch     : {trainer.best_epoch}".ljust(62) + "\u2551")
print("\u2551" + f"  Training Time  : {timedelta(seconds=int(total_time))}".ljust(62) + "\u2551")
print("\u2551" + f"  Smoke Test     : {CONFIG.get('smoke_test', False)}".ljust(62) + "\u2551")
print("\u2560" + "\u2550" * 62 + "\u2563")
print("\u2551" + "  Test Set Metrics".ljust(62) + "\u2551")
print("\u2560" + "\u2550" * 62 + "\u2563")
for metric_name, value in test_metrics.items():
    line = f"  {metric_name:<22s}: {value:.4f}"
    print("\u2551" + line.ljust(62) + "\u2551")
print("\u2560" + "\u2550" * 62 + "\u2563")
print("\u2551" + "  Normalization".ljust(62) + "\u2551")
print("\u2551" + f"  Source: {CONFIG['normalization']}".ljust(62) + "\u2551")
print("\u2551" + f"  Mean  : {[round(v, 4) for v in NORM_MEAN]}".ljust(62) + "\u2551")
print("\u2551" + f"  Std   : {[round(v, 4) for v in NORM_STD]}".ljust(62) + "\u2551")
print("\u2560" + "\u2550" * 62 + "\u2563")
print("\u2551" + "  Next Steps".ljust(62) + "\u2551")
print("\u2551" + "  1. Set smoke_test=False for full training".ljust(62) + "\u2551")
print("\u2551" + "  2. Compare with OMNet-V2 (multi-scale + attention)".ljust(62) + "\u2551")
print("\u2551" + "  3. Swap backbone to Vision Transformer".ljust(62) + "\u2551")
print("\u2551" + "  4. Integrate Proxy Anchor Loss via extract_features()".ljust(62) + "\u2551")
print("\u255a" + "\u2550" * 62 + "\u255d")

if CONFIG.get('smoke_test'):
    print("\n\u26a0\ufe0f  This was a SMOKE TEST (2 epochs).")
    print("   To run the full experiment:")
    print("   1. Set CONFIG['smoke_test'] = False")
    print("   2. Optionally adjust CONFIG['training']['epochs']")
    print("   3. Re-run all cells from Section 13 onward")
else:
    print("\n\u2705 Full experiment complete! Results saved to:", OUTPUT_DIR)

print("\n\U0001f52c OMNet \u2014 Research notebook execution complete.")